# 107 - Full Trade Stream: Real-Time Trade Analysis

Subscribe to real-time trades for a watchlist of stocks and a chain of
options, then analyze the flow: volume by symbol, large blocks, price
momentum, and put/call premium.

Streaming uses a **push-callback** model. You open a session, register a
callback, and subscribe with typed `Contract` values. The SDK invokes
your callback once per decoded event. Events are typed objects
(`Trade`, `Quote`, `ContractAssigned`, ...) narrowed with `isinstance`;
fields are plain attributes.

**Requires an active streaming session.** Real-time streaming needs a
market data subscription with real-time access, and trades flow only
during market hours (09:30 - 16:00 ET). The callback runs on the
dispatch thread - keep it cheap.

```
pip install thetadatadx[pandas] matplotlib ipywidgets
```

In [ ]:
import time
import threading
from collections import deque, defaultdict
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from thetadatadx import (
    Config, Contract, Credentials, Client,
    Trade, ContractAssigned, LoginSuccess,
)

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

## Part 1: Open the Streaming Session

A thread-safe buffer collects decoded events from the callback thread; the
analysis cells drain it on the main thread.

In [ ]:
creds = Credentials.from_file("creds.txt")
config = Config.production()
client = Client(creds, config)

_buffer = deque(maxlen=200_000)
_lock = threading.Lock()
_login = threading.Event()


def on_event(event):
    # Dispatch thread - keep it cheap.
    if isinstance(event, LoginSuccess):
        _login.set()
    with _lock:
        _buffer.append(event)


def drain():
    with _lock:
        items = list(_buffer)
        _buffer.clear()
    return items


session = client.streaming(on_event)
session.__enter__()

if _login.wait(timeout=5.0):
    print("Streaming session established.")
else:
    print("No login acknowledgement yet - the streaming feed may be unavailable.")
print(client)

## Part 2: Subscribe and Collect Decoded Trades

Subscribe to trade streams for a watchlist. Each `Trade` event carries
`price`, `size`, `exchange`, `condition`, `ms_of_day`, `sequence`, and a
`contract` with the resolved `symbol`.

In [ ]:
WATCHLIST = ["AAPL", "MSFT", "NVDA", "TSLA", "SPY", "QQQ", "AMD", "AMZN", "META", "GOOG"]

drain()  # clear startup events
for sym in WATCHLIST:
    client.stream.subscribe(Contract.stock(sym).trade())
    print(f"  Subscribed {sym}")
print(f"\nSubscribed to {len(WATCHLIST)} stock trade streams.")

In [ ]:
# Collect decoded trade events for 30 seconds.
COLLECT_SECONDS = 30

print(f"Collecting decoded trades for {COLLECT_SECONDS}s across {len(WATCHLIST)} symbols...")
time.sleep(COLLECT_SECONDS)

events = drain()
trades = [
    {
        "symbol": e.contract.symbol,
        "price": e.price,
        "size": e.size,
        "exchange": e.exchange,
        "condition": e.condition,
        "ms_of_day": e.ms_of_day,
        "sequence": e.sequence,
        "date": e.date,
    }
    for e in events if isinstance(e, Trade)
]
other = len(events) - len(trades)

print(f"\nCollected {len(trades):,} decoded trades in {COLLECT_SECONDS}s")
print(f"Throughput: {len(trades) / COLLECT_SECONDS:.1f} trades/sec")
if other:
    print(f"Other events: {other}")

df_trades = pd.DataFrame(trades)
if not df_trades.empty:
    df_trades["time_str"] = pd.to_timedelta(df_trades["ms_of_day"], unit="ms").apply(
        lambda td: f"{int(td.total_seconds()//3600):02d}:"
                   f"{int((td.total_seconds()%3600)//60):02d}:"
                   f"{td.total_seconds()%60:06.3f}"
    )
    df_trades["notional"] = df_trades["price"] * df_trades["size"]
    print(f"\nColumns: {list(df_trades.columns)}")
    display(df_trades[["time_str", "symbol", "price", "size",
                       "exchange", "condition"]].tail(20))
else:
    print("No trades received (market may be closed or feed unavailable).")

## Part 3: Real-Time Trade Analysis

Volume by symbol, large-block detection, price momentum, and buy/sell
imbalance.

In [ ]:
# Volume by symbol (bar chart) + large-block detector.
BLOCK_THRESHOLD = 1000  # shares

if not df_trades.empty:
    vol_by_sym = (
        df_trades.groupby("symbol")["size"].sum()
        .sort_values(ascending=False).head(10)
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    vol_by_sym.plot(kind="bar", ax=ax, color="#1f77b4", edgecolor="white")
    ax.set_title("Total Streamed Volume by Symbol")
    ax.set_ylabel("Shares")
    ax.set_xlabel("")
    ax.bar_label(ax.containers[0], fmt="{:,.0f}", fontsize=9)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    blocks = df_trades[df_trades["size"] > BLOCK_THRESHOLD].copy()
    print(f"\nLarge blocks (>{BLOCK_THRESHOLD:,} shares): {len(blocks):,} trades")
    if not blocks.empty:
        print(f"Total block volume: {blocks['size'].sum():,.0f} shares")
        print(f"Total block notional: ${blocks['notional'].sum():,.2f}")
        display(
            blocks.nlargest(15, "notional")[
                ["time_str", "symbol", "price", "size", "notional", "exchange", "condition"]
            ].reset_index(drop=True)
        )
    else:
        print("No large blocks detected in the sample window.")
else:
    print("No trade data available.")

In [ ]:
# Price momentum + buy/sell imbalance for the most active symbol.
if not df_trades.empty:
    most_active = df_trades.groupby("symbol")["size"].sum().idxmax()
    sym_trades = df_trades[df_trades["symbol"] == most_active].sort_values("ms_of_day").copy()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(sym_trades["ms_of_day"] / 3_600_000, sym_trades["price"],
            linewidth=0.7, color="#1f77b4", alpha=0.9)
    ax.set_title(f"{most_active} Price from the Trade Tape (sample window)")
    ax.set_xlabel("Hour (ET)")
    ax.set_ylabel("Price ($)")
    plt.tight_layout()
    plt.show()

    # Buy/sell imbalance via the tick rule:
    # uptick -> buy aggressor, downtick -> sell aggressor,
    # zero-tick -> carry the previous classification forward.
    side = []
    prev_price, prev_side = None, "buy"
    for p in sym_trades["price"]:
        if prev_price is not None:
            if p > prev_price:
                prev_side = "buy"
            elif p < prev_price:
                prev_side = "sell"
        side.append(prev_side)
        prev_price = p

    sym_trades["side"] = side
    buy_vol = sym_trades.loc[sym_trades["side"] == "buy", "size"].sum()
    sell_vol = sym_trades.loc[sym_trades["side"] == "sell", "size"].sum()
    total = buy_vol + sell_vol

    print(f"\n{most_active} Buy/Sell Imbalance (tick rule):")
    if total:
        print(f"  Buy volume:  {buy_vol:>12,} ({buy_vol/total*100:.1f}%)")
        print(f"  Sell volume: {sell_vol:>12,} ({sell_vol/total*100:.1f}%)")
        print(f"  Net:         {buy_vol - sell_vol:>+12,} shares")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Buy", "Sell"], [buy_vol, sell_vol],
           color=["#2ecc71", "#e74c3c"], edgecolor="white")
    ax.set_title(f"{most_active} Volume by Aggressor Side")
    ax.set_ylabel("Shares")
    ax.bar_label(ax.containers[0], fmt="{:,.0f}", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("No trade data available.")

## Part 4: Subscribe to Option Trades

Option trade streaming is per-contract: build a `Contract.option(...)` and
subscribe to its trade stream. We pick the nearest expiration and a spread
of strikes around ATM. `strike` is a dollar string; `right` is `"C"` or
`"P"`.

In [ ]:
OPT_UNDERLYING = "SPY"

# Spot and nearest upcoming expiration.
spot = client.historical.stock_snapshot_ohlc([OPT_UNDERLYING])[0].close
print(f"{OPT_UNDERLYING} spot: ${spot:.2f}")

expirations = client.historical.option_list_expirations(OPT_UNDERLYING).to_list()
today = datetime.now().date().isoformat()
expirations = [e for e in expirations if e > today]
nearest_exp = expirations[0]
dte = (datetime.strptime(nearest_exp, "%Y-%m-%d") - datetime.now()).days
print(f"Nearest expiration: {nearest_exp} ({dte} DTE)")

# Strikes near ATM (dollar strings).
all_strikes = client.historical.option_list_strikes(OPT_UNDERLYING, nearest_exp).to_list()
strike_pairs = sorted(((s, float(s)) for s in all_strikes), key=lambda x: x[1])
atm_idx = min(range(len(strike_pairs)), key=lambda i: abs(strike_pairs[i][1] - spot))

STRIKE_RANGE = 5
window = strike_pairs[max(0, atm_idx - STRIKE_RANGE):atm_idx + STRIKE_RANGE + 1]

opt_contracts = []
drain()
for strike_str, strike_val in window:
    for right in ("C", "P"):
        contract = Contract.option(
            OPT_UNDERLYING, expiration=nearest_exp, strike=strike_str, right=right)
        client.stream.subscribe(contract.trade())
        opt_contracts.append((strike_str, strike_val, right))
        print(f"  {OPT_UNDERLYING} {nearest_exp} {strike_val:.0f}{right}")

print(f"\nSubscribed to {len(opt_contracts)} option trade streams.")

In [ ]:
# Collect decoded option + stock trade events for 30 seconds.
OPT_COLLECT_SECONDS = 30

print(f"Collecting decoded trades for {OPT_COLLECT_SECONDS}s (stocks + options)...")
time.sleep(OPT_COLLECT_SECONDS)

events = drain()
stream_trades = [e for e in events if isinstance(e, Trade)]
print(f"Total decoded trade events: {len(stream_trades):,}")
print(f"Throughput: {len(stream_trades) / OPT_COLLECT_SECONDS:.1f} trades/sec")

## Part 5: Option Snapshots - Unusual Activity, P/C Ratio, Premium Flow

Pull the latest trade snapshot for each contract we subscribed to, then
compute notional, the put/call ratio, and net premium flow. Snapshot
methods are keyword-only and return a list of typed rows.

In [ ]:
opt_rows = []
for strike_str, strike_val, right in opt_contracts:
    try:
        snap = client.historical.option_snapshot_trade(
            OPT_UNDERLYING, expiration=nearest_exp, strike=strike_str, right=right)
    except Exception:
        snap = None
    if snap:
        t = snap[0]
        opt_rows.append({
            "underlying": OPT_UNDERLYING,
            "expiration": nearest_exp,
            "strike": strike_val,
            "right": right,
            "price": t.price,
            "size": t.size,
        })

df_opt = pd.DataFrame(opt_rows)

if not df_opt.empty:
    df_opt["notional"] = df_opt["price"] * df_opt["size"] * 100  # 100 shares / contract
    print(f"Option trade snapshots: {len(df_opt)} contracts with activity")
    display(
        df_opt[["underlying", "expiration", "strike", "right", "price", "size", "notional"]]
        .sort_values("notional", ascending=False).reset_index(drop=True)
    )

    UNUSUAL_THRESHOLD = 50_000  # $50k notional
    unusual = df_opt[df_opt["notional"] > UNUSUAL_THRESHOLD]
    print(f"\n=== UNUSUAL ACTIVITY (notional > ${UNUSUAL_THRESHOLD:,}) ===")
    print(f"Flagged contracts: {len(unusual)}")
    if not unusual.empty:
        display(
            unusual[["underlying", "strike", "right", "price", "size", "notional"]]
            .sort_values("notional", ascending=False).reset_index(drop=True)
        )

    calls = df_opt[df_opt["right"] == "C"]
    puts = df_opt[df_opt["right"] == "P"]
    call_vol = int(calls["size"].sum())
    put_vol = int(puts["size"].sum())
    pc_ratio = put_vol / call_vol if call_vol > 0 else float("inf")

    print(f"\n=== PUT/CALL RATIO ===")
    print(f"  Call volume: {call_vol:>8,} contracts")
    print(f"  Put volume:  {put_vol:>8,} contracts")
    print(f"  P/C ratio:   {pc_ratio:.3f}")

    call_premium = calls["notional"].sum()
    put_premium = puts["notional"].sum()
    net = call_premium - put_premium

    print(f"\n=== PREMIUM FLOW ===")
    print(f"  Call premium: ${call_premium:>14,.2f}")
    print(f"  Put premium:  ${put_premium:>14,.2f}")
    print(f"  Net (C - P):  ${net:>+14,.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(["Calls", "Puts"], [call_vol, put_vol],
                color=["#2ecc71", "#e74c3c"], edgecolor="white")
    axes[0].set_title(f"{OPT_UNDERLYING} Put/Call Volume (P/C = {pc_ratio:.2f})")
    axes[0].set_ylabel("Contracts")
    axes[0].bar_label(axes[0].containers[0], fmt="{:,.0f}")

    axes[1].bar(["Call $", "Put $"], [call_premium, put_premium],
                color=["#2ecc71", "#e74c3c"], edgecolor="white")
    axes[1].set_title(f"{OPT_UNDERLYING} Premium Flow (Net ${net:+,.0f})")
    axes[1].set_ylabel("Premium ($)")
    axes[1].bar_label(axes[1].containers[0], fmt="${:,.0f}")

    plt.tight_layout()
    plt.show()
else:
    print("No option trade snapshots returned (market may be closed or feed unavailable).")

## Part 6: Export and Clean Up

In [ ]:
# Save collected trades to CSV.
if not df_trades.empty:
    csv_path = f"stock_trades_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_trades[["time_str", "symbol", "price", "size",
               "exchange", "condition", "sequence"]].to_csv(csv_path, index=False)
    print(f"Stock trades saved: {csv_path} ({len(df_trades):,} rows)")

if not df_opt.empty:
    opt_csv = f"option_trades_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df_opt.to_csv(opt_csv, index=False)
    print(f"Option trades saved: {opt_csv} ({len(df_opt):,} rows)")

# Close the streaming session (stops the dispatch thread and drains).
session.__exit__(None, None, None)
print(client)
print("\nClean shutdown complete.")

---

**Previous:** [106 - Live Option Chain](106_live_option_chain.ipynb)
**Back to start:** [101 - Getting Started](101_getting_started.ipynb)